# Prepared-state reflections

Adapted from Qualtran v0.7.0 `reflections/reflections.ipynb`, copyright 2023 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Sequence
from typing import cast

import torch
from IPython import display as notebook_display

from flagquantum import Circuit
from flagquantum.algorithms.primitives.alias_preparation import (
    source_dense_alias_preparation_cost,
)
from flagquantum.algorithms.primitives.alias_sampling import alias_sampling_table
from flagquantum.algorithms.primitives.identity_prepare import IdentityPrepare
from flagquantum.algorithms.primitives.prepared_state_reflection import (
    PreparedStateReflection,
)
from flagquantum.algorithms.primitives.symbolic_alias_preparation import (
    AliasPreparationShape,
)
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost,
    estimate_component_resources,
)
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signatures(items: Sequence[tuple[str, Signature]]) -> None:
    for name, signature in items:
        display(SVG(render_call_graph_svg(export_signature_graph_dot(signature, name))))


def show_graph(graph: SymbolicCallGraph[str]) -> None:
    display(
        SVG(
            render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))
        )
    )
    display(graph.leaf_counts)


## PreparedStateReflection

For a preparation P and scalar g, the reflection is g P(I - 2|0><0|)P†.
The native compiler emits inverse preparation, zero-selective phase, scalar phase,
and preparation in that order. An optional binary control turns the scalar into
a relative phase. Selection and preparation junk retain their explicit ownership.
The mathematical inverse is requested through `make_on(adjoint=True)`.
The locked source declaration returns itself for every coefficient, including
nonreal coefficients. That original defect remains independently tested.

The examples preserve the five equal weights at precision .01 and the controlled
zero reflection over register widths 1/2/3 with coefficient -1.


In [ ]:
# PreparedStateReflection owns native declarations and composes existing binders.


### Example Instances

In [ ]:
table = alias_sampling_table([1] * 5, precision=0.01)
prepare_gate = AliasPreparationShape.from_table(table)
refl_using_prep = PreparedStateReflection(prepare_gate)


In [ ]:
refl_around_zero = PreparedStateReflection.around_zero(
    (1, 2, 3), global_phase=-1, control_value=1
)


#### Graphical Signature

In [ ]:
show_signatures(
    [
        ("Prepared-state reflection", refl_using_prep.signature),
        ("Controlled zero reflection", refl_around_zero.signature),
    ]
)


### Call Graph

In [ ]:
# Preserve the original one-level calls, including distinct PREPARE directions.
reflection_calls = {
    "Alias PREPARE": 1,
    "Alias PREPARE adjoint": 1,
    "MultiControlZ(0,0,0)": 1,
    "X": 2,
}
refl_using_prep_graph = symbolic_call_graph(
    "Prepared-state reflection",
    {"Prepared-state reflection": reflection_calls},
    max_depth=1,
)
show_graph(refl_using_prep_graph)
prepare_cost = source_dense_alias_preparation_cost(table)
modeled_resources = estimate_component_resources(
    refl_using_prep.resource_model(prepare_cost, prepare_cost)
)
display(modeled_resources)


## IdentityPrepare

Identity signal preparation preserves ordered selection registers. It supplies
zero-state reflections and block-encoding signal states. The 10/4/1 factory has
fifteen logical wires and zero QEC model cost. Explicit native identity expansion
and source SDK identity elision have separate primitive inventories.


In [ ]:
# IdentityPrepare shares the original ordered register declarations.


### Example Instances

In [ ]:
prepare_identity = IdentityPrepare.from_bitsizes((10, 4, 1))


#### Graphical Signature

In [ ]:
show_signatures([("Identity signal preparation", prepare_identity.signature)])


### Call Graph

In [ ]:
identity_calls = {
    f"Identity({r.bitsize})": 1 for r in prepare_identity.selection_registers
}
prepare_identity_graph = symbolic_call_graph(
    "Identity signal preparation",
    {"Identity signal preparation": identity_calls},
    max_depth=1,
)
show_graph(prepare_identity_graph)


## Native append and independent resource accounts
The one-bit fixture has one unitary branch and borrows a clean phase anchor. The model is queried independently from its emitted primitives. Larger reflections use the shared measured predicate and explicit classical slots.


In [ ]:
small = PreparedStateReflection.around_zero((1,), global_phase=-1)
op = small.make_on(selection={"reg0_": (0,)}, reflection_workspace=(1,))
circuit = Circuit(2, dtype=torch.complex128).h(0)
circuit.append(op)
measured_primitives = dict(Counter(g.name for g in op.instructions))
small_model = estimate_component_resources(
    small.resource_model(ComponentCost("I", 1), ComponentCost("I", 1))
)
display(measured_primitives)
display(small_model)
